# Green SM Ads Optimization — Kaggle full v1

Attach only `green-sm-sim-full` and `green-sm-full-v1-checkpoint`, enable a GPU, then run all cells. This trains a frozen-Foundation CTR ranker and exports label-free validation/test predictions.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/kaggle/working/repo')
if REPO.exists(): raise RuntimeError('Fresh Kaggle session required')
subprocess.run(['git','clone','--branch','feat/foundation-encoder','https://github.com/huongiang171/green-sm-foundation-pretraining.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],cwd=REPO,check=True)
print(subprocess.check_output(['git','log','-1','--oneline'],cwd=REPO,text=True))

In [ ]:
from pathlib import Path
import hashlib, json, torch, yaml
INPUT = Path('/kaggle/input')
def root(name, required):
    found=[p for p in INPUT.rglob(name) if p.is_dir() and (p/required).is_file()]
    assert len(found)==1, found
    return found[0]
CORPUS = root('green_sm_sim_full_v1','green_sm_sim_linked_v1/ads_opportunities/manifest.json')
CKPT = root('green_sm_full_v1_checkpoint','checkpoint_manifest.json')
LINKED = CORPUS/'green_sm_sim_linked_v1'
selection=json.loads((CKPT/'checkpoint_manifest.json').read_text())
assert selection['selected_kind'] in {'best','last'}
for p in [LINKED/'ads_opportunities/manifest.json', LINKED/'foundation/manifest.json', LINKED/'reports/data_quality_report.json', CKPT/selection['checkpoint'], CKPT/'vocabulary.json']:
    assert p.is_file(), p
assert torch.cuda.is_available(), 'Enable GPU Accelerator'
print(torch.cuda.get_device_name(0), LINKED, CKPT/selection['checkpoint'])
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

In [ ]:
with open(REPO/'configs/ad_optimization.yaml') as f: config=yaml.safe_load(f)
run=Path('/kaggle/working/artifacts/ad_optimization/ads-full-v1-r1')
if run.exists(): raise RuntimeError(f'Choose new run id: {run}')
ads=LINKED/'ads_opportunities'; foundation=LINKED/'foundation'
config['data'].update({'opportunities_root':str(ads),'manifest_path':str(ads/'manifest.json'),'foundation_root':str(foundation),'foundation_manifest_path':str(foundation/'manifest.json'),'quality_report_path':str(LINKED/'reports/data_quality_report.json'),'expected_ads_manifest_sha256':sha(ads/'manifest.json'),'expected_foundation_manifest_sha256':sha(foundation/'manifest.json')})
config['checkpoint'].update({'package_root':str(CKPT),'expected_selected_checkpoint_sha256':sha(CKPT/selection['checkpoint'])})
config['ranker'].update({'run_dir':str(run),'expected_run_manifest_sha256':'0'*64})
config['training'].update({'device':'cuda','batch_size':32,'epochs':20})
RUNTIME=Path('/kaggle/working/ad_optimization_full_v1.yaml')
with open(RUNTIME,'w') as f: yaml.safe_dump(config,f,sort_keys=False)
print(RUNTIME.read_text())

In [ ]:
subprocess.run([sys.executable,'-m','src.downstream.ad_optimization','--config',str(RUNTIME),'--mode','validate'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','src.downstream.ad_optimization','--config',str(RUNTIME),'--mode','train'],cwd=REPO,check=True)

In [ ]:
config=yaml.safe_load(RUNTIME.read_text())
config['ranker']['expected_run_manifest_sha256']=sha(run/'run_manifest.json')
RUNTIME.write_text(yaml.safe_dump(config,sort_keys=False))
for split in ('validation','test'):
    subprocess.run([sys.executable,'-m','src.downstream.ad_optimization','--config',str(RUNTIME),'--mode','predict','--split',split],cwd=REPO,check=True)
print(json.loads((run/'run_manifest.json').read_text()))

In [ ]:
import shutil
archive=shutil.make_archive('/kaggle/working/green_sm_ads_full_v1_r1','zip',root_dir=run)
print('Download:',archive)